In [6]:
!pip install scikit-learn

  Using cached scikit_learn-1.8.0-cp311-cp311-macosx_12_0_arm64.whl.metadata (11 kB)
  Using cached scipy-1.17.1-cp311-cp311-macosx_14_0_arm64.whl.metadata (62 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
Using cached scikit_learn-1.8.0-cp311-cp311-macosx_12_0_arm64.whl (8.1 MB)
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
Using cached scipy-1.17.1-cp311-cp311-macosx_14_0_arm64.whl (20.3 MB)
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [scikit-learn]0m 3/4 [scikit-learn]


In [7]:
import pandas as pd
import numpy as np

athletes = pd.read_csv("../data/athlete_events.csv")
regions = pd.read_csv("../data/noc_regions.csv")

df = athletes.merge(regions, on="NOC", how="left")

df = df[df["Season"] == "Summer"].copy()

def country_group(row):
    if row["NOC"] == "USA":
        return "United States"
    elif row["NOC"] in ["RUS", "URS", "EUN"]:
        return "Russia/USSR"
    elif row["NOC"] == "CHN":
        return "China"
    else:
        return "Other"

df["country_group"] = df.apply(country_group, axis=1)

df = df[df["country_group"].isin(["United States", "Russia/USSR", "China"])]

df["medaled"] = df["Medal"].notna().astype(int)

In [ ]:
panel = (
    df.groupby(["Year", "Sport", "country_group"])["medaled"]
    .sum()
    .reset_index()
)

panel["won_any_medal"] = (panel["medaled"] > 0).astype(int)

panel = panel.sort_values(["country_group", "Sport", "Year"])
panel["prior_medals"] = (
    panel.groupby(["country_group", "Sport"])["medaled"]
    .shift(1)
    .fillna(0)
)

In [8]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

X = panel[["Year", "Sport", "country_group", "prior_medals"]]
y = panel["won_any_medal"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=3120, stratify=y
)

preprocess = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), ["Sport", "country_group"]),
        ("num", "passthrough", ["Year", "prior_medals"])
    ]
)

model = Pipeline([
    ("prep", preprocess),
    ("logit", LogisticRegression(max_iter=1000))
])

model.fit(X_train, y_train)

pred = model.predict(X_test)
prob = model.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test, pred))
print("AUC:", roc_auc_score(y_test, prob))

Accuracy: 0.724025974025974
AUC: 0.7708940719144801


/Users/leoqian/ORIE3120/ORIE3120-Group18/venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
